# NB_Silver_Product_Category_Translation — Bronze to Silver Transformation

## Section 1 — Purpose, Business Context & Verified Profile

- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.product_category_name_translation` → OneLake Shortcut (`Files/bronze/catalog/product_category_name_translation/`).
- **Silver Target:** `silver.product_category_name_translation` (Managed Delta Table) + `silver._quarantine_product_category_translation` (Conditional Quarantine).
- **Grain:** One row per Portuguese category name (`product_category_name`, `71` rows).
- **Verified Bronze Profile (`71` rows × `2` columns):**
  - `0` NULLs, `0` exact duplicates, `0` duplicate `product_category_name`, `0` duplicate `product_category_name_english`, `0` leading/trailing whitespace, `0` uppercase characters, `0` non-ASCII characters.
  - **Source Spelling Quirks (Preserved & Documented):** `casa_conforto` → `home_confort` (vs `casa_conforto_2` → `home_comfort_2`), and `artes_e_artesanato` → `arts_and_craftmanship`.
  - **Coverage Gap vs `silver.products` (`73` non-null categories):** `2` Portuguese categories (`pc_gamer` [3 products] and `portateis_cozinha_e_preparadores_de_alimentos` [10 products]) are absent from this `71`-row reference table; `NB_Silver_Products.ipynb` handles both directly in `product_category_name_english`.
  - **Path Rule:** Read only the main `Files/bronze/catalog/product_category_name_translation/...` snapshot file; never read `bronze/_validation/...`.

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until Dry Run validation passes.

## Section 2 — Configuration & Safety Controls

Centralizes the relative OneLake Bronze shortcut path, Silver/Quarantine Managed Table targets, lineage constants, and `WRITE_ENABLED = False` safety gate.

In [ ]:
# Section 2: Configuration & Safety Controls
from pyspark.sql import functions as F

WRITE_ENABLED = False          # SAFETY LOCK: flip to True only after Dry Run review
RUN_PRODUCTS_COVERAGE_CHECK = True

BRONZE_PATH = "Files/bronze/catalog/product_category_name_translation/product_category_name_translation_20261121_013318_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.product_category_name_translation"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_product_category_translation"
SILVER_PRODUCTS_TABLE = f"{SILVER_SCHEMA}.products"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.product_category_name_translation"

EXPECTED_BRONZE_COLS = [
    "product_category_name",
    "product_category_name_english",
]

print(f"Configured BRONZE_PATH       : {BRONZE_PATH}")
print(f"Configured SILVER_TARGET     : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET : {QUARANTINE_TARGET_TABLE}")
print(f"WRITE_ENABLED                : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze Parquet Ingestion

Reads the `71`-row Bronze Parquet snapshot directly with PySpark and attaches `_source_file` lineage.

In [ ]:
# Section 3: Read Bronze Parquet & Inspect Schema
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PATH)
    .withColumn("_source_file", F.input_file_name())
)

bronze_count = df_bronze_raw.count()
print(f"Bronze Translation Row Count: {bronze_count:,} (Expected: 71)")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(10))

## Stage 1 — Schema Contract & Raw Null Baseline

Verifies that `product_category_name` and `product_category_name_english` are present, checks for unexpected extra columns, and counts raw `NULL`s.

In [ ]:
# Stage 1: Schema Contract & Baseline Profiling
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

raw_nulls = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
print("Stage 1 Schema Validation: PASSED")
display(raw_nulls)

## Stage 2 — Deterministic Key & Label Standardization

Applies `F.lower(F.trim(...))` to both `product_category_name` (join key) and `product_category_name_english` so joins with `silver.products` are deterministic.

In [ ]:
# Stage 2: Standardization
df_std = (
    df_bronze
    .withColumn("product_category_name", F.lower(F.trim(F.col("product_category_name").cast("string"))))
    .withColumn("product_category_name_english", F.lower(F.trim(F.col("product_category_name_english").cast("string"))))
)

print(f"Stage 2 complete. Distinct PT categories: {df_std.select('product_category_name').distinct().count()}")
print(f"Distinct EN categories: {df_std.select('product_category_name_english').distinct().count()}")

## Stage 3 & 4 — PK Uniqueness, Quarantine Routing & `silver.products` Coverage Audit

1. Enforces `1:1` uniqueness on `product_category_name` and non-empty strings on both columns, routing any bad row to `df_quarantine`.
2. If `silver.products` already exists in the Lakehouse, reports the `2` Portuguese categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) present in `silver.products` but absent from this `71`-row lookup.

In [ ]:
# Stage 3 & 4: PK Check, Quarantine Routing & Products Coverage Audit
exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()

dup_pk_df = (
    df_std.filter(F.col("product_category_name").isNotNull())
    .groupBy("product_category_name").count().filter(F.col("count") > 1)
    .select("product_category_name").withColumn("_dup_pk", F.lit(True))
)
dup_pk_cnt = dup_pk_df.count()

df_checked = (
    df_std
    .join(dup_pk_df, on="product_category_name", how="left")
    .withColumn("_dup_pk", F.coalesce(F.col("_dup_pk"), F.lit(False)))
    .withColumn(
        "dq_rejection_reason",
        F.concat_ws(
            " | ",
            F.when(F.col("product_category_name").isNull() | (F.length("product_category_name") == 0), F.lit("NULL_OR_BLANK_PT_CATEGORY")),
            F.when(F.col("product_category_name_english").isNull() | (F.length("product_category_name_english") == 0), F.lit("NULL_OR_BLANK_EN_CATEGORY")),
            F.when(F.col("_dup_pk"), F.lit("DUPLICATE_PT_CATEGORY_PK")),
        )
    )
)

df_valid = df_checked.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_checked.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(*EXPECTED_BRONZE_COLS, "dq_rejection_reason", "_source_file")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Exact Duplicates : {exact_dup_count} | Duplicate PKs : {dup_pk_cnt}")
print(f"Valid Rows       : {valid_count:,} (Expected: 71)")
print(f"Quarantined Rows : {quarantine_count:,} (Expected: 0)")

if RUN_PRODUCTS_COVERAGE_CHECK and spark.catalog.tableExists(SILVER_PRODUCTS_TABLE):
    untranslated_df = (
        spark.table(SILVER_PRODUCTS_TABLE)
        .select("product_category_name")
        .filter(F.col("product_category_name").isNotNull())
        .distinct()
        .join(df_valid.select("product_category_name").distinct(), on="product_category_name", how="left_anti")
    )
    print(f"Categories in {SILVER_PRODUCTS_TABLE} absent from 71-row lookup: {untranslated_df.count()} (Expected: 2)")
    display(untranslated_df)
else:
    print(f"Coverage audit vs {SILVER_PRODUCTS_TABLE} skipped (table not yet written or disabled).")

## Stage 5 — Lineage Enrichment & Reconciliation

Projects the `6`-column Silver contract (`2` core columns + `4` standardized lineage columns) and asserts `bronze_count == silver_count + quarantine_count`.

In [ ]:
# Stage 5: Lineage & Reconciliation
df_silver = (
    df_valid
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "product_category_name",
        "product_category_name_english",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)
df_silver.printSchema()
print(f"Stage 5 Reconciliation PASSED: bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")

## Section 5 — Guarded Delta Write (`silver.product_category_name_translation`)

Creates `silver` schema if missing and persists `silver.product_category_name_translation` strictly when `WRITE_ENABLED = True`.

In [ ]:
# Section 5: Guarded Delta Write
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification

Reads back `silver.product_category_name_translation` when `WRITE_ENABLED = True` and confirms `71` rows and `0` duplicate keys.

In [ ]:
# Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("product_category_name").count().filter(F.col("count") > 1).count()
    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate PK found: {chk_dup}"
    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}")
    display(df_check.orderBy("product_category_name").limit(10))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")